In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

from pyspark.sql.functions import *
import pyspark.sql.functions as F

StatementMeta(, 4b2d91df-fd86-42e2-9c68-f8e902501f60, 3, Finished, Available, Finished, False)

In [3]:

silver_vehicle = spark.read.table('silver_vehicle')
silver_supply_chain = spark.read.table('silver_supply_chain')
silver_quality = spark.read.table('silver_quality')
silver_process = spark.read.table('silver_process')
silver_manufacturing = spark.read.table('silver_manufacturing')
silver_fact_event = spark.read.table('silver_fact_event')

StatementMeta(, 4b2d91df-fd86-42e2-9c68-f8e902501f60, 5, Finished, Available, Finished, False)

In [4]:
display(silver_supply_chain)

StatementMeta(, 4b2d91df-fd86-42e2-9c68-f8e902501f60, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1241b2b-f24e-4413-9ae2-c17bbbbb1a4c)

In [5]:
display(silver_vehicle)

StatementMeta(, 4b2d91df-fd86-42e2-9c68-f8e902501f60, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c8afeb5-046f-4945-b11c-5833fbf36bb4)

# Production Summary

In [7]:
%%sql

CREATE OR REPLACE TABLE gold_production_summary AS 
SELECT
    p.order_date AS Date_production,
    m.plant_id AS plant_id,
    v.car_brand AS car_brand,

    SUM(f.quantity) AS total_quantity,
    AVG(q.quality_score) AS avg_quality_score,
    AVG(CASE WHEN q.defect_flag = TRUE THEN 0 ELSE 1 END) AS defect_rate,
    AVG(p.lead_time_days) AS avg_lead_time_days

FROM silver_fact_event f
JOIN silver_vehicle v ON f.vehicle_key = v.vehicle_key
JOIN silver_manufacturing m ON f.manufacturing_key = m.manufacturing_key
JOIN silver_process p ON f.process_key = p.process_key
JOIN silver_quality q ON f.quality_key = q.quality_key
GROUP BY p.order_date,
        m.plant_id,
        v.car_brand,
        f.quantity,
        q.defect_flag,
        p.lead_time_days;




SELECT * from gold_production_summary 


StatementMeta(, 4b2d91df-fd86-42e2-9c68-f8e902501f60, 11, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 970 rows and 7 fields>

# ML

**Data Augmentaion**

**Features col**

In [ ]:
'''
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml.pipeline import Pipeline


# Load weekly gold table
gold_df = spark.table("gold_production_summary")

gold_target = 100000
gold_current = gold_df.count()

multiplier = int(gold_target/ gold_current) + 1
# Add realistic noise to break perfect synthetic patterns
gold_noisy = (
    gold_df
    .withColumn("temp",
                F.explode(F.array([F.lit(x) for x in range (multiplier)]))

    )
    .drop("temp")
    .withColumn("defect_rate", F.col("defect_rate") * (F.rand() + F.randn() * 0.1))
    .withColumn("avg_quality_score", F.col("avg_quality_score") * (1 + F.randn() * 0.05))
    .withColumn("total_quantity",F.round( F.col("total_quantity") * (1 + F.randn() * 0.1)).cast("int"))
    .withColumn("avg_lead_time_days", F.round(F.col("avg_lead_time_days") * (1 + F.randn() * 0.05)).cast("int"))
)

gold_noisy.limit(100000)
# Clip defect_rate to [0,1]
gold_noisy = (
    gold_noisy
    # ---- defect_rate cleanup ----
    .withColumn(
        "defect_rate",
        F.round(
            F.when(F.col("defect_rate") == 0,
                   F.rand() * (0.3 - 0.1) + 0.1)
             .when(F.col("defect_rate") > 1,
                   F.rand() * (0.9 - 0.7) + 0.7)
             .when(F.col("defect_rate") < 0,
                   F.rand() * (0.2 - 0.1) + 0.1)
             .otherwise(F.col("defect_rate")),
            2
        )
    )

    # ---- avg_quality_score cleanup ----
    .withColumn(
        "avg_quality_score",
        F.round(
            F.when(F.col("avg_quality_score") > 100,
                   F.rand() * (99 - 85) + 85)
             .when(F.col("avg_quality_score") < 0,
                   F.rand() * (80 - 70) + 70)
             .otherwise(F.col("avg_quality_score")),
            2
        )
    )

   
)
    

#gold_noisy.write.mode("overwrite").saveAsTable("gold_noisy")

#display("save as table sucessfully! ", gold_noisy)
gold_df_production = gold_noisy.withColumn(
    "label",
    when(
        (col("defect_rate") > 0.4) |
        (col("avg_quality_score") < 85) |
        (col("avg_lead_time_days") > 7),
        1
    ).otherwise(0)
)


# Numeric features stay numeric
feature_cols = [
    "plant_id_cf",
    "car_brand_cf",
    "total_quantity",
    "avg_lead_time_days",
    "avg_quality_score",
    "defect_rate"
]

# Index ONLY categorical columns
indexers = [
    StringIndexer(inputCol="plant_id", outputCol="plant_id_cf"),
    StringIndexer(inputCol="car_brand", outputCol="car_brand_cf")
    
]



assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=10,
    maxIter=60,
    maxBins=1000,
    stepSize=0.02
)

pipeline = Pipeline(stages=indexers + [assembler, gbt])


gold_df_production.count()
display(gold_df_production)

from pyspark.ml.evaluation import BinaryClassificationEvaluator

train , test = gold_df_production.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train)
prediction = model.transform(test)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction"
)

auc = evaluator.evaluate(prediction)
print("AUC", auc)


from pyspark.ml.pipeline import PipelineModel

local_path = "abfss://cardataset@onelake.dfs.fabric.microsoft.com/Gold_LH.Lakehouse/Files/gold_production_summary_model1"

model = model.write().overwrite().save(local_path)

load_model = PipelineModel.load(local_path)
import pandas as pd 
import matplotlib.pyplot as plt 

# Extract features importance 
importances = load_model.stages[-1].featureImportances.toArray()

# Match them to featues names
feature_name = assembler.getInputCols()

df_imp = pd.DataFrame({
    "feature": feature_name,
    "importance":importances
}).sort_values("importance", ascending = False)

# Plot 

plt.figure(figsize=(10,6))
plt.barh(df_imp["feature"], df_imp["importance"])
plt.title("GBT Feature Importances")
plt.xlabel("Importance")
plt.show()

from pyspark.ml.functions import vector_to_array
from sklearn.metrics import roc_curve, auc
from matplotlib import pyplot as plt

pdf = prediction.select(
    vector_to_array("probability")[1].alias("prop"),
    "label"
).toPandas()

fpr, tpr, _ = roc_curve(pdf["label"], pdf["prop"])

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label = f"AUC = {roc_auc: .3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()
'''

# Engine Performance

In [9]:
%%sql

CREATE OR REPLACE TABLE gold_engine_performance AS
SELECT
    v.car_brand AS car_brand,
    v.engine_type AS engine_type,
    v.car_model AS car_model,
    v.body_style AS body_style,
    v.color AS color,
    v.horsepower AS horsepower,
    v.fuel_type AS fuel_type,
    v.model_year AS model_year,
    AVG(v.fuel_efficiency_l_100km) AS avg_fuel_efficiency,
    AVG(v.co2_emission_g_km) AS avg_co2_emission,
    AVG(v.Weight_kg) AS avg_weight,
    AVG(CASE WHEN q.defect_flag = true THEN 1 ELSE 0 END) AS defect_rate,
    v.vehicle_key AS vehicle_id

FROM silver_fact_event f
JOIN silver_vehicle v ON f.vehicle_key = v.vehicle_key
JOIN silver_quality q ON f.quality_key = q.quality_key 
GROUP BY 1, 2, 3,5,6, 7,8,v.vehicle_key,v.body_style;
    


select * from gold_engine_performance 


StatementMeta(, 6f3331cb-b6d8-46ad-8f57-f244fd6caf53, 14, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 970 rows and 13 fields>

# Supplier Performance

In [13]:
%%sql
CREATE OR REPLACE TABLE gold_supplier_performance AS 
SELECT 
    sc.supplier_id,
    SUM(f.quantity) AS  total_parts_delivered,
    AVG(p.lead_time_days) AS avg_lead_time,
    AVG(CASE WHEN q.defect_flag = true THEN 1 ELSE 0 END) AS defect_rate,
    AVG(CASE WHEN p.status = 'delayed' THEN 1 ELSE 0 END) AS on_time_delivery_rate

FROM silver_fact_event f 
JOIN silver_supply_chain sc ON f.supply_chain_key = sc.supply_chain_key
JOIN  silver_process p ON f.process_key = p.process_key
JOIN  silver_quality q ON f.quality_key = q.quality_key
GROUP BY 1; 

select * from gold_supplier_performance



StatementMeta(, 6f3331cb-b6d8-46ad-8f57-f244fd6caf53, 20, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 10 rows and 5 fields>

# Logistic Summary 

In [5]:
%%sql 
CREATE OR REPLACE TABLE gold_logistic_summary AS 
SELECT 
    sc.route AS route,
    sc.transport_mode AS transport_mode,
    AVG(p.lead_time_days) AS avg_lead_time,
    SUM(f.quantity) AS total_quantity_moved,
    AVG(CASE WHEN p.status = 'delayed' THEN 1 ELSE 0 END) AS delay_rate

FROM silver_fact_event f
left JOIN silver_supply_chain sc ON f.supply_chain_key = sc.supply_chain_key
left JOIN silver_process p ON  f.process_key = p.process_key

GROUP BY 1,2;
 
SELECT * FROM gold_logistic_summary


StatementMeta(, 6f3331cb-b6d8-46ad-8f57-f244fd6caf53, 8, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 18 rows and 5 fields>

# Quality Summary

In [16]:
%%sql
CREATE OR REPLACE TABLE gold_quality_summary AS
SELECT 
    v.car_brand AS car_brand,
    v.car_model AS car_model,
    m.plant_id AS plant_id,
    AVG(q.quality_score) AS avg_quality_score,
    AVG(CASE WHEN q.defect_flag = true THEN 1 ELSE 0 END) AS defect_rate

FROM silver_fact_event f
JOIN silver_vehicle v ON f.vehicle_key = v.vehicle_key
JOIN silver_manufacturing m ON  f.manufacturing_key = m.manufacturing_key
JOIN silver_quality q ON f.quality_key = q.quality_key

GROUP BY 1,2,3;

SELECT * FROM gold_quality_summary


StatementMeta(, f076c8b6-c338-414e-85d4-aa674526a1cc, 28, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 406 rows and 5 fields>

# join tables for US Dataset

In [ ]:
%%sql
 
SELECT 
    fs.OrderID,
    fs.OrderDate,
    fs.Sales,
    fs.Quantity,
    fs.Discount,
    dp.CarMaker,
    dp.CarModel,
    spec.EngineType,
    spec.BodyStyle,
    dc.CustomerName,
    dc.City,
    dc.State,
    ds.SupplierName,
    cal.Year,
    cal.MonthName,
    cal.Quarter
FROM silver_Fact_Sales fs
LEFT JOIN silver_Dim_Product  dp ON fs.ProductID  =  dp.ProductID
LEFT JOIN silver_Product_Spec spec ON dp.ProductID = spec.ProductID
LEFT JOIN silver_Dim_Customer dc ON fs.CustomerID = dc.CustomerID
LEFT JOIN silver_Dim_Supplier ds ON fs.SupplierID = ds.SupplierID
LEFT JOIN silver_calendar cal ON fs.OrderDate = cal.Date


StatementMeta(, c53d3108-ea6e-4dc3-8174-a0335aabf8bf, -1, Cancelled, , Cancelled, True)

# Save the table in Lake house

In [23]:
gold_production_summary = spark.sql("SELECT * FROM Silver_LH.dbo.gold_production_summary LIMIT 5")
display(gold_production_summary)

StatementMeta(, f076c8b6-c338-414e-85d4-aa674526a1cc, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81620055-b523-4732-9c3c-513c41f6b210)

In [25]:
gold_production_summary.write.mode('overwrite').option("mergeSchema","true").format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Gold_LH.Lakehouse/Tables/dbo/gold_production_summary')

StatementMeta(, f076c8b6-c338-414e-85d4-aa674526a1cc, 38, Finished, Available, Finished, False)

In [ ]:
gold_engine_performance.write.mode('overwrite').format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Gold_LH.Lakehouse/Tables/dbo/gold_engine_performance')

StatementMeta(, f076c8b6-c338-414e-85d4-aa674526a1cc, -1, Cancelled, , Cancelled, True)

In [ ]:
gold_supplier_performance.write.mode('overwrite').format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Gold_LH.Lakehouse/Tables/dbo/gold_supplier_performance')

StatementMeta(, f076c8b6-c338-414e-85d4-aa674526a1cc, -1, Cancelled, , Cancelled, True)

In [ ]:
gold_quality_summary.write.mode('overwrite').format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Gold_LH.Lakehouse/Tables/dbo/gold_quality_summary')

StatementMeta(, f076c8b6-c338-414e-85d4-aa674526a1cc, -1, Cancelled, , Cancelled, True)

In [ ]:
gold_logistic_summary.write.mode('overwrite').format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Gold_LH.Lakehouse/Tables/dbo/gold_logistic_summary')

StatementMeta(, f076c8b6-c338-414e-85d4-aa674526a1cc, -1, Cancelled, , Cancelled, True)